In [ ]:
import torch
import torch.nn as nn
import math


class PositionalEncoding(nn.Module):
    """Standard sinusoidal positional encoding."""

    def __init__(self, d_model, max_len=100):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0)  # (1, max_len, d_model)
        self.register_buffer('pe', pe)

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]


class CrossAttentionLayer(nn.Module):
    """
    Cross-attention layer: query sequence attends to key/value sequence.
    
    Allows the before encoder to see the after sequence and vice versa.
    Uses standard multi-head attention with residual connection and layer norm.
    """

    def __init__(self, d_model, nhead, dropout=0.2):
        super(CrossAttentionLayer, self).__init__()
        self.cross_attn = nn.MultiheadAttention(
            embed_dim=d_model, num_heads=nhead, dropout=dropout, batch_first=True
        )
        self.norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, query_seq, kv_seq, query_mask=None, kv_mask=None):
        """
        Args:
            query_seq: (batch, seq_len_q, d_model) - sequence that asks questions
            kv_seq: (batch, seq_len_kv, d_model) - sequence that provides answers
            query_mask: (batch, seq_len_q) - 1=valid, 0=padding for query
            kv_mask: (batch, seq_len_kv) - 1=valid, 0=padding for key/value
        """
        # Convert to attention mask format (True = IGNORE)
        key_padding_mask = None
        if kv_mask is not None:
            key_padding_mask = (kv_mask == 0)

        attn_output, _ = self.cross_attn(
            query=query_seq,
            key=kv_seq,
            value=kv_seq,
            key_padding_mask=key_padding_mask
        )

        # Residual connection + layer norm
        output = self.norm(query_seq + self.dropout(attn_output))
        return output


class AttentionPooling(nn.Module):
    """
    Attention-based pooling with a learned query vector.
    
    Instead of mean pooling (which treats all timesteps equally),
    this learns to focus on the most relevant observations.
    The learned query asks: "given what I need to predict, 
    which observations matter most?"
    """

    def __init__(self, d_model, nhead=4, dropout=0.2):
        super(AttentionPooling, self).__init__()
        # Learned query vector (single token that aggregates the sequence)
        self.query = nn.Parameter(torch.randn(1, 1, d_model) * 0.02)
        self.attn = nn.MultiheadAttention(
            embed_dim=d_model, num_heads=nhead, dropout=dropout, batch_first=True
        )
        self.norm = nn.LayerNorm(d_model)

    def forward(self, seq, mask=None):
        """
        Args:
            seq: (batch, seq_len, d_model)
            mask: (batch, seq_len) - 1=valid, 0=padding
        
        Returns:
            pooled: (batch, d_model)
        """
        batch_size = seq.size(0)

        # Expand learned query to batch size
        query = self.query.expand(batch_size, -1, -1)  # (batch, 1, d_model)

        # Mask for key padding
        key_padding_mask = None
        if mask is not None:
            key_padding_mask = (mask == 0)

        # Single query attends to all valid positions
        attn_output, _ = self.attn(
            query=query,
            key=seq,
            value=seq,
            key_padding_mask=key_padding_mask
        )

        # (batch, 1, d_model) -> (batch, d_model)
        pooled = self.norm(attn_output.squeeze(1))
        return pooled


class TemporalTransformer_GapFilling_V3(nn.Module):
    """
    Temporal Transformer V3 for Sentinel-2 gap-filling.
    
    Changes from V2:
    1. Cross-attention between before/after encoders
       - Before encoder can see what happened after the gap
       - After encoder can see what happened before the gap
       - Enables the model to understand the full temporal context
    
    2. Attention-based pooling (replaces mean pooling)
       - Learned query focuses on most relevant observations
       - Near observations get higher weight when available
       - Graceful fallback to seasonal patterns for long gaps
    
    Architecture:
        before_seq → project → pos_encode → self_attn_before → cross_attn(before→after) → attn_pool →
                                                                                                        concat → dense → output
        after_seq  → project → pos_encode → self_attn_after  → cross_attn(after→before) → attn_pool →
    """

    def __init__(self,
                 input_size=13,
                 d_model=128,
                 nhead=8,
                 num_layers=4,
                 num_cross_layers=2,
                 dim_feedforward=512,
                 current_meteo_size=8,
                 static_size=9,
                 output_size=2,
                 dropout=0.2):
        super(TemporalTransformer_GapFilling_V3, self).__init__()

        self.d_model = d_model

        # Input projection
        self.input_projection = nn.Linear(input_size, d_model)

        # Positional encoding
        self.pos_encoder = PositionalEncoding(d_model, max_len=100)

        # Self-attention encoders (same as V2)
        encoder_layer_before = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True
        )
        self.transformer_before = nn.TransformerEncoder(
            encoder_layer_before, num_layers=num_layers
        )

        encoder_layer_after = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward,
            dropout=dropout, batch_first=True
        )
        self.transformer_after = nn.TransformerEncoder(
            encoder_layer_after, num_layers=num_layers
        )

        # Cross-attention layers (NEW)
        # Multiple layers for deeper cross-sequence understanding
        self.cross_attn_before_to_after = nn.ModuleList([
            CrossAttentionLayer(d_model, nhead, dropout)
            for _ in range(num_cross_layers)
        ])
        self.cross_attn_after_to_before = nn.ModuleList([
            CrossAttentionLayer(d_model, nhead, dropout)
            for _ in range(num_cross_layers)
        ])

        # Attention-based pooling (NEW - replaces mean pooling)
        self.attn_pool_before = AttentionPooling(d_model, nhead=4, dropout=dropout)
        self.attn_pool_after = AttentionPooling(d_model, nhead=4, dropout=dropout)

        # Dense layers (same structure as V2)
        concat_size = 2 * d_model + current_meteo_size + static_size

        self.fc1 = nn.Linear(concat_size, 256)
        self.bn1 = nn.BatchNorm1d(256)
        self.dropout1 = nn.Dropout(dropout)

        self.fc2 = nn.Linear(256, 128)
        self.bn2 = nn.BatchNorm1d(128)
        self.dropout2 = nn.Dropout(dropout)

        self.fc3 = nn.Linear(128, 64)
        self.bn3 = nn.BatchNorm1d(64)
        self.dropout3 = nn.Dropout(dropout)

        self.output = nn.Linear(64, output_size)
        self.relu = nn.ReLU()

    def forward(self, before_seq, before_mask, after_seq, after_mask,
                current_meteo, static_features):
        """
        Forward pass with cross-attention and attention pooling.
        
        Args:
            before_seq: (batch, MAX_N, input_size)
            before_mask: (batch, MAX_N) - 1=valid, 0=padding
            after_seq: (batch, MAX_N, input_size)
            after_mask: (batch, MAX_N) - 1=valid, 0=padding
            current_meteo: (batch, 8)
            static_features: (batch, 9)
        
        Returns:
            output: (batch, 2) - predicted B4, B8
        """
        # 1. Project and add positional encoding
        before_proj = self.pos_encoder(self.input_projection(before_seq))
        after_proj = self.pos_encoder(self.input_projection(after_seq))

        # 2. Self-attention (same as V2)
        before_pad_mask = (before_mask == 0)
        after_pad_mask = (after_mask == 0)

        before_encoded = self.transformer_before(
            before_proj, src_key_padding_mask=before_pad_mask
        )
        after_encoded = self.transformer_after(
            after_proj, src_key_padding_mask=after_pad_mask
        )

        # 3. Cross-attention (NEW)
        # Before attends to after, and after attends to before
        # Multiple layers for iterative refinement
        before_cross = before_encoded
        after_cross = after_encoded

        for cross_b2a, cross_a2b in zip(
            self.cross_attn_before_to_after,
            self.cross_attn_after_to_before
        ):
            # Before looks at after
            before_cross_new = cross_b2a(
                query_seq=before_cross,
                kv_seq=after_cross,
                query_mask=before_mask,
                kv_mask=after_mask
            )
            # After looks at before
            after_cross_new = cross_a2b(
                query_seq=after_cross,
                kv_seq=before_cross,
                query_mask=after_mask,
                kv_mask=before_mask
            )
            before_cross = before_cross_new
            after_cross = after_cross_new

        # 4. Attention pooling (NEW - replaces mean pooling)
        before_pooled = self.attn_pool_before(before_cross, before_mask)
        after_pooled = self.attn_pool_after(after_cross, after_mask)

        # 5. Concatenate with meteo and static
        combined = torch.cat([
            before_pooled,
            after_pooled,
            current_meteo,
            static_features
        ], dim=1)

        # 6. Dense layers
        x = self.relu(self.bn1(self.fc1(combined)))
        x = self.dropout1(x)
        x = self.relu(self.bn2(self.fc2(x)))
        x = self.dropout2(x)
        x = self.relu(self.bn3(self.fc3(x)))
        x = self.dropout3(x)

        return self.output(x)


def count_parameters(model):
    """Count trainable parameters."""
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


if __name__ == "__main__":
    """Test the V3 model architecture."""

    print("=" * 80)
    print("TEMPORAL TRANSFORMER V3 MODEL TEST")
    print("  - Cross-attention between before/after")
    print("  - Attention-based pooling (replaces mean pooling)")
    print("=" * 80)

    model_v3 = TemporalTransformer_GapFilling_V3(
        input_size=13,
        d_model=128,
        nhead=8,
        num_layers=4,
        num_cross_layers=2,
        dim_feedforward=512,
        current_meteo_size=8,
        static_size=9,
        output_size=2,
        dropout=0.2
    )

    print(f"\n📊 Model Architecture:")
    print(model_v3)

    print(f"\n📊 V3 Parameters: {count_parameters(model_v3):,}")

    # Test forward pass
    batch_size = 16
    max_n = 40

    before_seq = torch.randn(batch_size, max_n, 13)
    before_mask = torch.ones(batch_size, max_n)
    before_mask[:, :10] = 0  # First 10 are padding

    after_seq = torch.randn(batch_size, max_n, 13)
    after_mask = torch.ones(batch_size, max_n)
    after_mask[:, 30:] = 0  # Last 10 are padding

    current_meteo = torch.randn(batch_size, 8)
    static_features = torch.randn(batch_size, 9)

    print(f"\n🧪 Testing forward pass:")
    print(f"  Batch size: {batch_size}")
    print(f"  MAX_N: {max_n}")

    output = model_v3(before_seq, before_mask, after_seq, after_mask,
                      current_meteo, static_features)

    print(f"\n  Output shape: {output.shape}")
    print(f"  Output range: [{output.min().item():.4f}, {output.max().item():.4f}]")

    # Test with extreme masking (simulating large gap)
    print(f"\n🧪 Testing with extreme masking (gap 7 scenario):")
    before_mask_extreme = torch.ones(batch_size, max_n)
    before_mask_extreme[:, :10] = 0   # padding
    before_mask_extreme[:, -3:] = 0    # hide 3 nearest

    after_mask_extreme = torch.ones(batch_size, max_n)
    after_mask_extreme[:, 30:] = 0    # padding
    after_mask_extreme[:, :3] = 0      # hide 3 nearest

    output_extreme = model_v3(before_seq, before_mask_extreme, after_seq, after_mask_extreme,
                              current_meteo, static_features)
    print(f"  Output shape: {output_extreme.shape}")
    print(f"  Output range: [{output_extreme.min().item():.4f}, {output_extreme.max().item():.4f}]")
    print(f"  Any NaN: {torch.isnan(output_extreme).any().item()}")

    print(f"\n✅ V3 Model test successful!")
    print(f"{'=' * 80}")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, ConcatDataset
import numpy as np
from pathlib import Path
import json
from tqdm import tqdm
import time
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

print("=" * 80)
print("TEMPORAL TRANSFORMER TRAINING")
print("=" * 80)

# ============================================================================
# CONFIGURATION
# ============================================================================

SEQUENCE_DIR = Path("transformers_sequences")
CHECKPOINT_DIR = Path("checkpoints_transformer_v3_gap_aware_v2")
CHECKPOINT_DIR.mkdir(exist_ok=True)

BATCH_SIZE = 4096
LEARNING_RATE = 0.0001
NUM_EPOCHS = 50
EARLY_STOPPING_PATIENCE = 10

INPUT_SIZE = 13
D_MODEL = 128
NHEAD = 8
NUM_SELF_ATTN_LAYERS = 4
NUM_CROSS_ATTN_LAYERS = 2
DIM_FEEDFORWARD = 512
DROPOUT = 0.2

WEIGHT_B4 = 10.0
WEIGHT_B8 = 1.0

MAX_HIDE = 3

# New distribution: emphasise mid-range gaps (hide=1 is gap 2-3)
# Old: [0.40, 0.25, 0.20, 0.15]
# New: [0.25, 0.35, 0.25, 0.15]
GAP_WEIGHTS = [0.25, 0.35, 0.25, 0.15]

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
N_GPUS = torch.cuda.device_count() if torch.cuda.is_available() else 0

print(f"\n📋 Configuration:")
print(f"  Device: {DEVICE}")
if DEVICE.type == 'cuda':
    print(f"  GPUs available: {N_GPUS}")
    for i in range(N_GPUS):
        print(f"    GPU {i}: {torch.cuda.get_device_name(i)}")
    if N_GPUS > 1:
        print(f"  ✓ DataParallel enabled across {N_GPUS} GPUs")
        print(f"  Effective batch size: {BATCH_SIZE} x {N_GPUS} = {BATCH_SIZE * N_GPUS}")
print(f"  Batch size per GPU: {BATCH_SIZE}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"  Input size: {INPUT_SIZE}")
print(f"  Loss weights: B4={WEIGHT_B4}, B8={WEIGHT_B8}")
print(f"\n🎯 Gap-aware training v2:")
print(f"  MAX_HIDE: {MAX_HIDE}")
print(f"  Gap weights [hide 0/1/2/3]: {GAP_WEIGHTS}")
print(f"  (Old was [0.40, 0.25, 0.20, 0.15])")


# ============================================================================
# MEMORY-EFFICIENT DATASET
# ============================================================================

class SingleGridDataset(Dataset):
    def __init__(self, filepath):
        self.filepath = filepath
        data = np.load(filepath, mmap_mode='r')
        self.before_padded = data['before_padded']
        self.before_masks = data['before_masks']
        self.after_padded = data['after_padded']
        self.after_masks = data['after_masks']
        self.current_meteo = data['current_meteo']
        self.static_features = data['static_features']
        self.targets = data['targets']
        self.n_samples = len(self.targets)

    def __len__(self):
        return self.n_samples

    def __getitem__(self, idx):
        return {
            'before_seq': torch.from_numpy(self.before_padded[idx].copy()).float(),
            'before_mask': torch.from_numpy(self.before_masks[idx].copy()).float(),
            'after_seq': torch.from_numpy(self.after_padded[idx].copy()).float(),
            'after_mask': torch.from_numpy(self.after_masks[idx].copy()).float(),
            'current_meteo': torch.from_numpy(self.current_meteo[idx].copy()).float(),
            'static_features': torch.from_numpy(self.static_features[idx].copy()).float(),
            'target': torch.from_numpy(self.targets[idx].copy()).float()
        }


def load_multi_grid_dataset(sequence_dir, grid_ids, split_name):
    datasets = []
    total_sequences = 0

    print(f"\n📂 Loading {split_name.upper()} data (memory-mapped)...")
    for grid_id in grid_ids:
        filepath = sequence_dir / f"sequences_grid_{grid_id}_{split_name}.npz"
        if not filepath.exists():
            print(f"  ⚠️ {filepath.name} not found, skipping...")
            continue

        ds = SingleGridDataset(filepath)
        datasets.append(ds)
        total_sequences += len(ds)
        print(f"  ✓ {filepath.name}: {len(ds):,} sequences")

    combined = ConcatDataset(datasets)
    print(f"  ✓ Total: {total_sequences:,} sequences")
    return combined


# ============================================================================
# GAP-AWARE MASKING
# ============================================================================

def sample_gap_sizes(batch_size, max_hide, device, gap_weights):
    """
    Sample random gap sizes with configurable distribution.
    Before and after sampled independently for asymmetric gaps.
    """
    weights = torch.tensor(gap_weights, device=device)
    weights = weights[:max_hide + 1]
    weights = weights / weights.sum()

    hide_before = torch.multinomial(weights.expand(batch_size, -1), num_samples=1).squeeze(1)
    hide_after = torch.multinomial(weights.expand(batch_size, -1), num_samples=1).squeeze(1)

    return hide_before, hide_after


def apply_gap_masking(before_mask, after_mask, hide_before, hide_after):
    """Apply per-sample gap masking."""
    b_mask = before_mask.clone()
    a_mask = after_mask.clone()

    for i in range(before_mask.shape[0]):
        hb = hide_before[i].item()
        ha = hide_after[i].item()

        if hb > 0:
            b_mask[i, -hb:] = 0
        if ha > 0:
            a_mask[i, :ha] = 0

    return b_mask, a_mask


# ============================================================================
# LOSS AND EVALUATION
# ============================================================================

def weighted_mse_loss(pred, target, weight_b4=10.0, weight_b8=1.0):
    loss_b4 = nn.functional.mse_loss(pred[:, 0], target[:, 0])
    loss_b8 = nn.functional.mse_loss(pred[:, 1], target[:, 1])
    weighted_loss = weight_b4 * loss_b4 + weight_b8 * loss_b8
    return weighted_loss, loss_b4, loss_b8


def evaluate(model, dataloader, device, weight_b4, weight_b8):
    model.eval()

    all_preds = []
    all_targets = []
    total_loss = 0.0
    total_loss_b4 = 0.0
    total_loss_b8 = 0.0

    with torch.no_grad():
        for batch in tqdm(dataloader, desc="  Evaluating", leave=False):
            before_seq = batch['before_seq'].to(device)
            before_mask = batch['before_mask'].to(device)
            after_seq = batch['after_seq'].to(device)
            after_mask = batch['after_mask'].to(device)
            current_meteo = batch['current_meteo'].to(device)
            static_features = batch['static_features'].to(device)
            target = batch['target'].to(device)

            pred = model(before_seq, before_mask, after_seq, after_mask,
                         current_meteo, static_features)

            loss, loss_b4, loss_b8 = weighted_mse_loss(pred, target, weight_b4, weight_b8)
            total_loss += loss.item()
            total_loss_b4 += loss_b4.item()
            total_loss_b8 += loss_b8.item()

            all_preds.append(pred.cpu().numpy())
            all_targets.append(target.cpu().numpy())

    all_preds = np.concatenate(all_preds, axis=0)
    all_targets = np.concatenate(all_targets, axis=0)

    n_batches = len(dataloader)
    metrics = {
        'loss': total_loss / n_batches,
        'loss_b4': total_loss_b4 / n_batches,
        'loss_b8': total_loss_b8 / n_batches,
        'r2_b4': r2_score(all_targets[:, 0], all_preds[:, 0]),
        'r2_b8': r2_score(all_targets[:, 1], all_preds[:, 1]),
        'rmse_b4': np.sqrt(mean_squared_error(all_targets[:, 0], all_preds[:, 0])),
        'rmse_b8': np.sqrt(mean_squared_error(all_targets[:, 1], all_preds[:, 1])),
        'mae_b4': mean_absolute_error(all_targets[:, 0], all_preds[:, 0]),
        'mae_b8': mean_absolute_error(all_targets[:, 1], all_preds[:, 1])
    }

    return metrics


# ============================================================================
# GAP-AWARE TRAINING FUNCTION
# ============================================================================

def train_epoch_gap_aware(model, dataloader, optimizer, device, weight_b4, weight_b8,
                          epoch, max_hide, gap_weights):
    model.train()

    total_loss = 0.0
    total_loss_b4 = 0.0
    total_loss_b8 = 0.0

    pbar = tqdm(dataloader, desc=f"  Epoch {epoch + 1}")
    for batch in pbar:
        before_seq = batch['before_seq'].to(device)
        before_mask = batch['before_mask'].to(device)
        after_seq = batch['after_seq'].to(device)
        after_mask = batch['after_mask'].to(device)
        current_meteo = batch['current_meteo'].to(device)
        static_features = batch['static_features'].to(device)
        target = batch['target'].to(device)

        batch_size = before_seq.shape[0]

        # Sample and apply gap masking
        hide_before, hide_after = sample_gap_sizes(batch_size, max_hide, device, gap_weights)
        before_mask_gapped, after_mask_gapped = apply_gap_masking(
            before_mask, after_mask, hide_before, hide_after
        )

        pred = model(before_seq, before_mask_gapped, after_seq, after_mask_gapped,
                     current_meteo, static_features)

        loss, loss_b4, loss_b8 = weighted_mse_loss(pred, target, weight_b4, weight_b8)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()
        total_loss_b4 += loss_b4.item()
        total_loss_b8 += loss_b8.item()

        pbar.set_postfix_str(
            f'loss={loss.item():.4f} | B4={loss_b4.item():.4f} | B8={loss_b8.item():.4f}'
        )

    n_batches = len(dataloader)
    return total_loss / n_batches, total_loss_b4 / n_batches, total_loss_b8 / n_batches


# ============================================================================
# MAIN
# ============================================================================

def main():
    train_grids = [1, 2, 9, 10, 11, 12]
    val_grids = [4, 8]

    train_dataset = load_multi_grid_dataset(SEQUENCE_DIR, train_grids, 'train')
    val_dataset = load_multi_grid_dataset(SEQUENCE_DIR, val_grids, 'val')

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=0,
        pin_memory=True if DEVICE.type == 'cuda' else False
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0,
        pin_memory=True if DEVICE.type == 'cuda' else False
    )

    print(f"\n📊 Data loaded:")
    print(f"  Training: {len(train_dataset):,} sequences, {len(train_loader):,} batches")
    print(f"  Validation: {len(val_dataset):,} sequences, {len(val_loader):,} batches")

    # Verify
    sample = train_dataset[0]
    actual_input_size = sample['before_seq'].shape[-1]
    assert actual_input_size == INPUT_SIZE
    print(f"\n  ✓ Feature dimension: {actual_input_size}")
    print(f"  ✓ MAX_N: {sample['before_seq'].shape[0]}")

    # Create V3 model
    model = TemporalTransformer_GapFilling_V3(
        input_size=INPUT_SIZE,
        d_model=D_MODEL,
        nhead=NHEAD,
        num_layers=NUM_SELF_ATTN_LAYERS,
        num_cross_layers=NUM_CROSS_ATTN_LAYERS,
        dim_feedforward=DIM_FEEDFORWARD,
        current_meteo_size=8,
        static_size=9,
        output_size=2,
        dropout=DROPOUT
    ).to(DEVICE)

    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\n📊 V3 Model: {n_params:,} parameters")

    if DEVICE.type == 'cuda':
        print(f"  GPU memory: {torch.cuda.memory_allocated(0) / 1024**3:.2f} GB")

    optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min', factor=0.5, patience=5
    )

    history = {
        'train_loss': [], 'train_loss_b4': [], 'train_loss_b8': [],
        'val_loss': [], 'val_loss_b4': [], 'val_loss_b8': [],
        'val_r2_b4': [], 'val_r2_b8': [],
        'val_rmse_b4': [], 'val_rmse_b8': [],
        'val_mae_b4': [], 'val_mae_b8': []
    }

    best_val_loss = float('inf')
    patience_counter = 0

    print(f"\n{'=' * 80}")
    print("STARTING V3 GAP-AWARE v2 TRAINING")
    print(f"  Gap weights: {GAP_WEIGHTS}")
    print(f"{'=' * 80}")

    for epoch in range(NUM_EPOCHS):
        epoch_start = time.time()

        print(f"\nEpoch {epoch + 1}/{NUM_EPOCHS}")
        print("-" * 80)

        train_loss, train_loss_b4, train_loss_b8 = train_epoch_gap_aware(
            model, train_loader, optimizer, DEVICE,
            WEIGHT_B4, WEIGHT_B8, epoch, MAX_HIDE, GAP_WEIGHTS
        )

        print(f"\n  Validating (gap-1, no masking)...")
        val_metrics = evaluate(model, val_loader, DEVICE, WEIGHT_B4, WEIGHT_B8)

        scheduler.step(val_metrics['loss'])

        history['train_loss'].append(train_loss)
        history['train_loss_b4'].append(train_loss_b4)
        history['train_loss_b8'].append(train_loss_b8)
        history['val_loss'].append(val_metrics['loss'])
        history['val_loss_b4'].append(val_metrics['loss_b4'])
        history['val_loss_b8'].append(val_metrics['loss_b8'])
        history['val_r2_b4'].append(val_metrics['r2_b4'])
        history['val_r2_b8'].append(val_metrics['r2_b8'])
        history['val_rmse_b4'].append(val_metrics['rmse_b4'])
        history['val_rmse_b8'].append(val_metrics['rmse_b8'])
        history['val_mae_b4'].append(val_metrics['mae_b4'])
        history['val_mae_b8'].append(val_metrics['mae_b8'])

        epoch_time = time.time() - epoch_start

        print(f"\n  Results:")
        print(f"    Train Loss (gap-aware): {train_loss:.4f} (B4: {train_loss_b4:.4f}, B8: {train_loss_b8:.4f})")
        print(f"    Val Loss (gap-1):       {val_metrics['loss']:.4f} (B4: {val_metrics['loss_b4']:.4f}, B8: {val_metrics['loss_b8']:.4f})")
        print(f"    Val R² (gap-1):         B4={val_metrics['r2_b4']:.4f}, B8={val_metrics['r2_b8']:.4f}")
        print(f"    Val RMSE:               B4={val_metrics['rmse_b4']:.6f}, B8={val_metrics['rmse_b8']:.6f}")
        print(f"    Val MAE:                B4={val_metrics['mae_b4']:.6f}, B8={val_metrics['mae_b8']:.6f}")
        print(f"    Time: {epoch_time:.1f}s")

        if val_metrics['loss'] < best_val_loss:
            best_val_loss = val_metrics['loss']
            patience_counter = 0

            checkpoint = {
                'epoch': epoch + 1,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_loss': val_metrics['loss'],
                'val_r2_b4': val_metrics['r2_b4'],
                'val_r2_b8': val_metrics['r2_b8'],
                'history': history,
                'config': {
                    'input_size': INPUT_SIZE,
                    'd_model': D_MODEL,
                    'nhead': NHEAD,
                    'num_self_attn_layers': NUM_SELF_ATTN_LAYERS,
                    'num_cross_attn_layers': NUM_CROSS_ATTN_LAYERS,
                    'dim_feedforward': DIM_FEEDFORWARD,
                    'dropout': DROPOUT,
                    'current_meteo_size': 8,
                    'static_size': 9,
                    'output_size': 2,
                    'max_n': 40,
                    'max_hide': MAX_HIDE,
                    'gap_weights': GAP_WEIGHTS,
                    'version': 'v3_gap_aware_v2'
                }
            }

            torch.save(checkpoint, CHECKPOINT_DIR / 'best_model.pth')
            print(f"\n  ✓ Best model saved! (Val Loss: {best_val_loss:.4f})")
        else:
            patience_counter += 1
            print(f"\n  No improvement. Patience: {patience_counter}/{EARLY_STOPPING_PATIENCE}")

        if patience_counter >= EARLY_STOPPING_PATIENCE:
            print(f"\n⏹ Early stopping triggered after {epoch + 1} epochs")
            break

    torch.save({'model_state_dict': model.state_dict(), 'history': history},
               CHECKPOINT_DIR / 'final_model.pth')

    with open(CHECKPOINT_DIR / 'training_history_v3_gap_aware_v2.json', 'w') as f:
        json.dump(history, f, indent=2)

    print(f"\n{'=' * 80}")
    print("V3 GAP-AWARE v2 TRAINING COMPLETE!")
    print(f"{'=' * 80}")
    print(f"\n✓ Best validation loss: {best_val_loss:.4f}")
    print(f"✓ Models saved to: {CHECKPOINT_DIR}")


if __name__ == "__main__":
    main()

In [ ]:
import torch
import torch.nn as nn
import numpy as np
from pathlib import Path
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import json
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

# ============================================================================
# ⚙️ CONFIGURATION
# ============================================================================
SEQUENCE_DIR = Path("transformers_sequences")  # Same data as V2
CHECKPOINT_PATH = Path("checkpoints_transformer_v3_gap_aware_v2/best_model.pth")
BATCH_SIZE = 4096
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

TEST_GRIDS = [3, 13]

print("=" * 80)
print("TRANSFORMER V3 GAP-AWARE v2 - REALISTIC GAP POSITION ANALYSIS")
print("  Cross-attention + attention pooling")
print("=" * 80)
print(f"\nTest Grids: {TEST_GRIDS}")
print(f"Device: {DEVICE}")


# ============================================================================
# 📂 DATASET
# ============================================================================
class GapFillingDataset(Dataset):
    def __init__(self, sequence_dir, grid_ids, split_name):
        self.sequences = []
        print(f"\n📂 Loading {split_name.upper()} data...")
        for grid_id in grid_ids:
            filepath = sequence_dir / f"sequences_grid_{grid_id}_{split_name}.npz"
            if not filepath.exists():
                print(f"  ⚠️ {filepath.name} not found, skipping...")
                continue

            print(f"  Loading {filepath.name}...")
            data = np.load(filepath)
            self.sequences.append({
                'before_padded': torch.from_numpy(data['before_padded']).float(),
                'before_masks': torch.from_numpy(data['before_masks']).float(),
                'after_padded': torch.from_numpy(data['after_padded']).float(),
                'after_masks': torch.from_numpy(data['after_masks']).float(),
                'current_meteo': torch.from_numpy(data['current_meteo']).float(),
                'static_features': torch.from_numpy(data['static_features']).float(),
                'targets': torch.from_numpy(data['targets']).float()
            })
            print(f"    Loaded {len(data['targets']):,} sequences")

        print(f"\n  Concatenating...")
        self.before_padded = torch.cat([s['before_padded'] for s in self.sequences], dim=0)
        self.before_masks = torch.cat([s['before_masks'] for s in self.sequences], dim=0)
        self.after_padded = torch.cat([s['after_padded'] for s in self.sequences], dim=0)
        self.after_masks = torch.cat([s['after_masks'] for s in self.sequences], dim=0)
        self.current_meteo = torch.cat([s['current_meteo'] for s in self.sequences], dim=0)
        self.static_features = torch.cat([s['static_features'] for s in self.sequences], dim=0)
        self.targets = torch.cat([s['targets'] for s in self.sequences], dim=0)

        print(f"  ✓ Total sequences: {len(self):,}")
        print(f"  ✓ Feature size: {self.before_padded.shape[-1]}")
        print(f"  ✓ MAX_N: {self.before_padded.shape[1]}")

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, idx):
        return {
            'before_seq': self.before_padded[idx],
            'before_mask': self.before_masks[idx],
            'after_seq': self.after_padded[idx],
            'after_mask': self.after_masks[idx],
            'current_meteo': self.current_meteo[idx],
            'static_features': self.static_features[idx],
            'target': self.targets[idx]
        }


# ============================================================================
# 📋 GAP SCENARIOS
# ============================================================================
GAP_SCENARIOS = [
    (1,  "Gap 1",              "D, Y, D",                   0, 0),
    (2,  "Gap 2 (edge-after)", "D, X, Y, D",                1, 0),
    (3,  "Gap 2 (edge-before)","D, Y, X, D",                0, 1),
    (4,  "Gap 3 (center)",     "D, X, Y, X, D",             1, 1),
    (5,  "Gap 4 (pos 3/4)",    "D, X, X, Y, X, D",          2, 1),
    (6,  "Gap 4 (pos 2/4)",    "D, X, Y, X, X, D",          1, 2),
    (7,  "Gap 5 (center)",     "D, X, X, Y, X, X, D",       2, 2),
    (8,  "Gap 6 (pos 3/6)",    "D, X, X, Y, X, X, X, D",    2, 3),
    (9,  "Gap 6 (pos 4/6)",    "D, X, X, X, Y, X, X, D",    3, 2),
    (10, "Gap 7 (center)",     "D, X, X, X, Y, X, X, X, D", 3, 3),
]


# ============================================================================
# 🚀 MAIN EXECUTION
# ============================================================================
if __name__ == "__main__":

    # Load Model
    print(f"\n📂 Loading model from {CHECKPOINT_PATH}...")
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)

    if 'config' in checkpoint:
        config = checkpoint['config']
        print(f"  Config: {config}")
    else:
        config = {}

    model = TemporalTransformer_GapFilling_V3(
        input_size=config.get('input_size', 13),
        d_model=config.get('d_model', 128),
        nhead=config.get('nhead', 8),
        num_layers=config.get('num_self_attn_layers', 4),
        num_cross_layers=config.get('num_cross_attn_layers', 2),
        dim_feedforward=config.get('dim_feedforward', 512),
        current_meteo_size=config.get('current_meteo_size', 8),
        static_size=config.get('static_size', 9),
        output_size=config.get('output_size', 2),
        dropout=config.get('dropout', 0.2)
    ).to(DEVICE)

    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()
    print(f"  ✓ Model loaded from epoch {checkpoint.get('epoch', '?')}")
    print(f"  ✓ Parameters: {sum(p.numel() for p in model.parameters()):,}")
    if 'val_r2_b4' in checkpoint:
        print(f"  ✓ Training val R²: B4={checkpoint['val_r2_b4']:.4f}, B8={checkpoint['val_r2_b8']:.4f}")

    # Load Test Data
    test_ds = GapFillingDataset(SEQUENCE_DIR, TEST_GRIDS, 'test')
    loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    # Context stats
    before_valid = test_ds.before_masks.sum(dim=1)
    after_valid = test_ds.after_masks.sum(dim=1)
    print(f"\n📏 Context availability:")
    print(f"  BEFORE - Min: {before_valid.min():.0f}, Max: {before_valid.max():.0f}, Mean: {before_valid.mean():.1f}")
    print(f"  AFTER  - Min: {after_valid.min():.0f}, Max: {after_valid.max():.0f}, Mean: {after_valid.mean():.1f}")

    # Run scenarios
    print(f"\n{'=' * 110}")
    print("V3 GAP-AWARE v2 REALISTIC GAP POSITION ANALYSIS")
    print(f"{'=' * 110}")
    print(f"\nD = available data, X = gap (hidden), Y = target being predicted\n")

    header = (f"{'#':<3}| {'Description':<22}| {'Pattern':<34}| {'Band':<16}| "
              f"{'R2':<9}| {'Pearson':<9}| {'RMSE':<9}| {'MAE':<9}| {'Bias':<9}| {'Samples':<10}")
    print(header)
    print("-" * 110)

    all_results = []

    for scenario_id, desc, pattern, hide_before, hide_after in GAP_SCENARIOS:
        total_gap = hide_before + 1 + hide_after
        total_days = total_gap * 15

        # Warn about thin context
        if hide_before > 0 or hide_after > 0:
            remaining_before = before_valid - hide_before
            remaining_after = after_valid - hide_after
            thin = ((remaining_before < 1) | (remaining_after < 1)).sum().item()
            if thin > 0:
                pct = thin / len(test_ds) * 100
                if pct > 5:
                    print(f"  ⚠ Scenario {scenario_id}: {thin:,} seqs ({pct:.1f}%) will lose all context on a side")

        all_preds = []
        all_targets = []

        with torch.no_grad():
            for batch in tqdm(loader, leave=False, desc=f"  Scenario {scenario_id}"):
                b_seq = batch['before_seq'].clone().to(DEVICE)
                b_msk = batch['before_mask'].clone().to(DEVICE)
                a_seq = batch['after_seq'].clone().to(DEVICE)
                a_msk = batch['after_mask'].clone().to(DEVICE)
                meteo = batch['current_meteo'].to(DEVICE)
                stat = batch['static_features'].to(DEVICE)
                target = batch['target']

                if hide_before > 0:
                    b_msk[:, -hide_before:] = 0
                if hide_after > 0:
                    a_msk[:, :hide_after] = 0

                pred = model(b_seq, b_msk, a_seq, a_msk, meteo, stat)

                all_preds.append(pred.cpu().numpy())
                all_targets.append(target.numpy())

        all_preds = np.concatenate(all_preds, axis=0)
        all_targets = np.concatenate(all_targets, axis=0)

        # Filter NaN
        valid_mask = ~np.isnan(all_preds).any(axis=1) & ~np.isnan(all_targets).any(axis=1)
        n_nan = (~valid_mask).sum()
        if n_nan > 0:
            pct = n_nan / len(valid_mask) * 100
            print(f"  ⚠ Filtered {n_nan:,} NaN predictions ({pct:.1f}%)")
            all_preds = all_preds[valid_mask]
            all_targets = all_targets[valid_mask]

        all_preds = np.maximum(all_preds, 0.0)

        bands = {0: "Band 4 (Red)", 1: "Band 8 (NIR)"}

        for b_idx, b_name in bands.items():
            y_pred = all_preds[:, b_idx]
            y_true = all_targets[:, b_idx]

            r2 = r2_score(y_true, y_pred)
            rmse = np.sqrt(mean_squared_error(y_true, y_pred))
            mae = mean_absolute_error(y_true, y_pred)
            pearson = np.corrcoef(y_true, y_pred)[0, 1]
            bias = np.mean(y_pred - y_true)

            print(f"{scenario_id:<3}| {desc:<22}| {pattern:<34}| {b_name:<16}| "
                  f"{r2:.4f}  | {pearson:.4f}  | {rmse:.4f}  | {mae:.4f}  | "
                  f"{bias:>8.4f} | {len(y_true):,}")

            all_results.append({
                'scenario_id': scenario_id,
                'description': desc,
                'pattern': pattern,
                'hide_before': hide_before,
                'hide_after': hide_after,
                'total_gap_timesteps': total_gap,
                'total_gap_days': total_days,
                'band': b_name,
                'r2': float(r2),
                'pearson': float(pearson),
                'rmse': float(rmse),
                'mae': float(mae),
                'bias': float(bias),
                'n_samples': len(y_true)
            })

        print("-" * 110)

    # Summary
    print(f"\n{'=' * 110}")
    print("SUMMARY BY TOTAL GAP SIZE")
    print(f"{'=' * 110}")

    gap_sizes = sorted(set(r['total_gap_timesteps'] for r in all_results))
    for gap_size in gap_sizes:
        gap_results = [r for r in all_results if r['total_gap_timesteps'] == gap_size]
        print(f"\n  Gap {gap_size} timesteps ({gap_size * 15} days):")
        for r in gap_results:
            print(f"    {r['description']:<22} | {r['band']:<16} | R²={r['r2']:.4f} | "
                  f"Pearson={r['pearson']:.4f} | RMSE={r['rmse']:.4f}")

    # Save
    results_file = CHECKPOINT_PATH.parent / 'realistic_gap_results_v3_gap_aware_v2.json'
    with open(results_file, 'w') as f:
        json.dump(all_results, f, indent=2)
    print(f"\n\n📊 Results saved to: {results_file}")

    print(f"\n{'=' * 110}")
    print("✅ V3 GAP-AWARE v2 ANALYSIS COMPLETE!")
    print(f"{'=' * 110}")